# verify08b: 学習モデル → 全分岐予測 → ベクトル生成（検証入力）

`train/df_validation_input*.csv`（1行=1通報のワイド形式）を、verify07 の**学習モデル(A or B)**で全分岐推論し、
**年齢論理集合(機能1)**を反映して **~106長ベクトル**を作る。→ そのベクトルは verify09 の `vector_to_triage` でトリアージできる。

**流れ**: 検証CSV → `validation_infer`(列→ノード対応105 + モデル予測 + 年齢上書き) → ベクトル → `vector_to_triage(vector, age)` → 詳細レポート
- 列→ノード対応: `transition_diagram/validation_column_map.json`（文で解決済み・105列）
- 質問A: **MODE='A'** は文(=yaml質問)そのまま / **MODE='B'** は選択肢付き（node_questions_B）
- 保存モデル: `checkpoints/verify07_7_21_symptom_full_{A|B}/final_model/`


In [ ]:
# ===== 設定 =====
import os, sys, glob, json, csv
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODE       = 'A'                                   # 'A'(選択肢なし) / 'B'(選択肢あり)。学習に使ったモデルと必ず合わせる
VAL_CSV    = 'train/df_validation_input (1).csv'   # 検証入力
MAX_LENGTH = 160
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# final_model の探索（ローカル / Colab Drive の両対応）
_CANDS = [
    f'checkpoints/verify07_7_21_symptom_full_{MODE}/final_model',
    f'/content/drive/MyDrive/emergency_task/checkpoints/verify07_7_21_symptom_full_{MODE}/final_model',
]
MODEL_DIR = next((d for d in _CANDS if os.path.exists(os.path.join(d, 'config.json'))), None)
assert MODEL_DIR, f'final_model が見つかりません。verify07_{MODE} を学習してください。候補={_CANDS}'
print('MODE =', MODE, '| MODEL_DIR =', MODEL_DIR, '| DEVICE =', DEVICE)


In [ ]:
# ===== モデルロード & predict_fn =====
tok   = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE).eval()
print('num_labels =', model.config.num_labels)

@torch.no_grad()
def predict_fn(A: str, B: str) -> int:
    enc = tok(A, B, truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
    return int(model(**enc).logits.argmax(-1).cpu())


In [ ]:
# ===== 全行を推論 → ベクトル生成 =====
import importlib, validation_infer as vi, vector_triage as vt
importlib.reload(vi); importlib.reload(vt)

rows = vi.load_validation(VAL_CSV)
print(f'検証 {len(rows)} 行 / ベクトル長(NODE_ORDER)={len(vt.NODE_ORDER)}')

vectors, ages, diags = [], [], []
for row in rows:
    vec, age, info = vi.row_to_vector(row, predict_fn, mode=MODE)
    vectors.append(vec); ages.append(age); diags.append(info)

# ベクトルを保存（verify09 で読める形式）
os.makedirs('output', exist_ok=True)
out_vec = f'output/validation_vectors_{MODE}.csv'
with open(out_vec, 'w', encoding='utf-8-sig', newline='') as f:
    w = csv.writer(f)
    w.writerow(['id', 'age'] + vt.NODE_ORDER)
    for row, vec, age in zip(rows, vectors, ages):
        w.writerow([row.get('id'), age] + vec)
print('ベクトル保存 →', out_vec)


In [ ]:
# ===== ベクトル → トリアージ ＋ 詳細レポート（原因追跡）=====
from collections import Counter
results = []
for row, vec, age, info in zip(rows, vectors, ages, diags):
    res = vt.vector_to_triage(vec, age=age)
    results.append({'id': row.get('id'), 'age': age, 'main': res.main, 'sub1': res.sub1,
                    'common_completed': res.common_completed, 'common_stop': res.common_stop,
                    'completed': ','.join(res.completed_symptoms or []),
                    'furthest_broke': res.furthest_broke_symptom,
                    'report': res.report()})

# 先頭3件の詳細レポート
for r in results[:3]:
    print('='*60)
    print('id=%s age=%s' % (r['id'], r['age']))
    print(r['report'])

# サマリ
print('')
print('==== サマリ ====')
print('main分布 :', dict(Counter(r['main'] for r in results)))
cc = [r['common_completed'] for r in results]
print('共通フロー完了率 :', '%d/%d' % (sum(1 for x in cc if x), len(cc)))
print('共通が途切れた止まりノード上位 :', Counter(r['common_stop'] for r in results if not r['common_completed']).most_common(5))


In [ ]:
# ===== 原因追跡: 各ケースで「どのノードが何を予測したか」＋年齢上書き箇所 =====
CASE = 0    # 見たいケースの index
info = diags[CASE]; res = results[CASE]
print('id=%s age=%s / main=%s 共通完了=%s 止まり=%s' % (info['id'], info['age'], res['main'], res['common_completed'], res['common_stop']))
print('')
print('列別 予測（非該当0以外＋年齢上書き）:')
for d in info['diagnostics']:
    if d['code'] != 0 or d['src'] == 'age_logic':
        print('  %-36s code=%s  [%s]  (%s)' % (d['node'], d['code'], d['src'], d['col']))


## まとめ
- **08b = 学習モデル(A/B) + 年齢論理集合 → ベクトル生成**。列→ノードは文で105/105解決済み。
- 出力 `output/validation_vectors_{MODE}.csv`（id, age, 各ノードcode）は **verify09 の入力**にそのまま使える。
- **原因追跡**: `res.report()` で「共通/症候のどこで途切れたか」、最後のセルで「各ノードの予測code」を確認 → 誤りノードを特定できる。
- MODE は学習モデルと必ず一致させる（A↔A / B↔B）。既存の `verify08_年齢論理集合_検証.ipynb`(部品単体テスト) / `verify09`(ベクトル→トリアージ) と役割分担。
